# 05f - Task 3: LightGBM retune (round 4)

`05_tuning.ipynb` is left untouched as the historical record. This notebook re-searches
LightGBM properly, for two reasons established in notebook 09.

**Why re-search at all.** Notebook 09 showed that once predicted class balance is held
fixed, local CV ranks models *correctly* - LightGBM beat ElasticNet by +0.0189 on the
public leaderboard at matched share 0.4996, mirroring its +0.015 CV lead. CV gains now
transfer at roughly 1:1, so tuning is no longer a bet against the train-to-test shift.

**Why the old search was not enough.** 05 ran 8 random draws over a 7-dimensional space,
then a 3x3 grid on `learning_rate` x `num_leaves` with the other five knobs frozen at the
stage-1 winner - 26 configurations. And the winner sits at `min_child_samples=7` and
`num_leaves=149` against ranges of 5-60 and 15-150: **the search terminated on its own
boundary in two dimensions**, which is the classic signature of an optimum lying outside
the box that was searched. Section 3 widens exactly those two and adds four knobs 05
never touched.

**What makes a bigger search affordable:** the sparse CSR feature path added in
`src/data.py`. 01_eda measured the matrix at 98.6% sparse, so the dense float64 train
matrix is 800 MB of mostly zeros; CSR float32 cuts that ~50x and speeds up every fit.

**Budget:** time-boxed, not trial-count-boxed - see section 2.

## 0. Setup

In [1]:
# Reload src/ helpers on every cell execution. Round 4 edits src/data.py, src/tuning.py
# and src/ensemble.py while these notebooks are open, and a plain `import` caches the
# module in the kernel - so a fixed helper keeps failing with the OLD traceback until the
# kernel is restarted. With this, saving the .py is enough.
%load_ext autoreload
%autoreload 2

# Adds project root to path so `import src...` works from notebooks/.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import json
import joblib

from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import ParameterSampler
from lightgbm import LGBMClassifier

from src import paths, data, evaluation, tuning
from src.paths import FIGURES
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load features (sparse) + the locked split

`sparse=True` returns a float32 CSR matrix, cached to `data/processed/` on first call so
the 210 MB CSV parse happens once per machine rather than once per run. Row indexing
(`X[dev_idx]`) works exactly as before.

`data.check_sparse_path()` is the guard against the one failure mode this change could
introduce silently: a shifted column order or dropped row would train happily and just
score badly, with nothing raising. Run it once per machine.

In [2]:
print(data.check_sparse_path())  # asserts shape, ~98.6% sparsity, and agreement with the dense path

X, y, ids = data.load_train_features(sparse=True)
dev_idx = np.load(paths.DATA_PROCESSED / 'dev_idx.npy')
cv = evaluation.make_cv()
      
X_dev, y_dev = X[dev_idx], y[dev_idx]
print(f"dev {X_dev.shape}, {X_dev.nnz / (X_dev.shape[0] * X_dev.shape[1]):.4f} dense, "
      f"{X_dev.data.nbytes / 1e6:.0f} MB")

{'shape': (20000, 5000), 'density': 0.01357, 'max_abs_diff': 4.997482289104127e-09}
dev (16000, 5000), 0.0135 dense, 4 MB


## 2. Your assignment

**Change `ME` and nothing else.** Everything below derives from it.

The stage-1 space is split three ways by `learning_rate` band, equal in log space
(log10 from -2.000 to -0.523, divided in thirds). README.md already nominates
`learning_rate` as the axis to split on, because it interacts most with every other knob.
Every other dimension is identical for all three of us, so the union covers the full box
exactly once and nobody duplicates anyone else's trials.

| member | `OWNER` | `learning_rate` band | stage-2 seed |
|---|---|---|---|
| Jovyan | `jovyan_lr_lo` | 0.010 - 0.031 | 0 |
| Cliffton | `cliffton_lr_mid` | 0.031 - 0.097 | 1 |
| Brian Wong | `brian_lr_hi` | 0.097 - 0.300 | 2 |

**Budget: 2h00 for stage 1, 0h30 for stage 2.** `tuning.run_search` stops when the
wall clock is spent rather than after a fixed number of trials, because trial cost varies
several-fold across machines. Every trial is written the moment it finishes, so stopping
partway loses nothing - a slower laptop just contributes fewer trials to the merge.

In [3]:
ASSIGNMENTS = {
    "jovyan":   {"owner": "jovyan_lr_lo",    "lr": (0.010, 0.031), "seed": 0},
    "cliffton": {"owner": "cliffton_lr_mid", "lr": (0.031, 0.097), "seed": 1},
    "brian":    {"owner": "brian_lr_hi",     "lr": (0.097, 0.300), "seed": 2},
}

ME = "cliffton"  # <-- THE ONLY LINE TO CHANGE

mine = ASSIGNMENTS[ME]
OWNER, (LR_LO, LR_HI), STAGE2_SEED = mine["owner"], mine["lr"], mine["seed"]
STAGE1_BUDGET_S = 2 * 3600
STAGE2_BUDGET_S = 30 * 60

print(f"{ME}: OWNER={OWNER}, learning_rate in [{LR_LO}, {LR_HI}], stage-2 seed {STAGE2_SEED}")

cliffton: OWNER=cliffton_lr_mid, learning_rate in [0.031, 0.097], stage-2 seed 1


## 3. Stage 1 - the widened search space

Changes from 05, and why each one:

| knob | 05 range | here | why |
|---|---|---|---|
| `num_leaves` | 15-150 | **15-400** | 05's winner was 149, at the ceiling |
| `min_child_samples` | 5-60 | **2-60** | 05's winner was 7, near the floor |
| `min_child_weight` | not searched | 1e-3 - 10 (log) | LightGBM's other leaf regularizer, independent of `min_child_samples` |
| `subsample` | not searched | 0.6 - 1.0 | row bagging; never tried, and decorrelates trees the way `colsample_bytree` does for columns |
| `max_bin` | not searched | 63 / 127 / 255 | fewer bins regularizes *and* speeds up sparse input |
| `learning_rate` | 0.01-0.3 | same range, **split 3 ways** | carried, but now covered densely rather than by 8 draws |
| `n_estimators`, `colsample_bytree`, `reg_alpha`, `reg_lambda` | keep | keep | |

Fixed, as in 05: `class_weight="balanced"`, `random_state=42`, `verbose=-1`.
`subsample_freq=1` is fixed too - LightGBM ignores `subsample` entirely when the
frequency is 0, so searching the fraction without it would have been a silent no-op.

> **Model key.** Trials are written under `lightgbm_v2_stage1`, NOT `lightgbm_stage1`.
> `tuning.load_trials` globs on the key, so reusing 05's name would merge two different
> search spaces into one table and break notebook 05's reproducibility. Section 4 asserts
> this did not happen.

In [ ]:
param_dist = {
    "learning_rate": loguniform(LR_LO, LR_HI),      # your band only
    "n_estimators": randint(150, 900),
    "num_leaves": randint(15, 400),                 # widened: 05's winner hit 150
    "min_child_samples": randint(2, 60),            # widened: 05's winner hit the floor
    "min_child_weight": loguniform(1e-3, 10),       # new
    "subsample": uniform(0.6, 0.4),                 # new, loc=0.6 scale=0.4 -> [0.6, 1.0)
    "colsample_bytree": uniform(0.2, 0.8),          # widened low end; 05's winner was 0.52
    "max_bin": [63, 127, 255],                      # new
    "reg_alpha": uniform(0.0, 2.0),
    "reg_lambda": uniform(0.0, 2.0),
}

FIXED = dict(class_weight="balanced", subsample_freq=1, random_state=42,
             verbose=-1, n_jobs=-1)


def build_lgbm(params):
    '''Fresh estimator per trial - run_trial does not call set_params itself.'''
    return LGBMClassifier(**FIXED, **params)


# n_iter is an upper bound only; the budget is what actually stops the loop.
sampler = ParameterSampler(param_dist, n_iter=400, random_state=hash(OWNER) % 2**31)

stage1_mine = tuning.run_search(
    "lightgbm_v2_stage1", OWNER, build_lgbm, sampler,
    X_dev, y_dev, cv, budget_seconds=STAGE1_BUDGET_S,
)
print(f"\n{len(stage1_mine)} trials this run; best {stage1_mine['mean'].max():.4f}"
      if len(stage1_mine) else "\nno trials completed")

[1] mean=0.7381 std=0.0061  (0s, 0.0/120 min, 1481116 trials/hr)
[2] mean=0.7409 std=0.0052  (0s, 0.0/120 min, 1683108 trials/hr)
[3] mean=0.7385 std=0.0064  (0s, 0.0/120 min, 1951184 trials/hr)
[4] mean=0.7388 std=0.0076  (0s, 0.0/120 min, 1975038 trials/hr)
[5] mean=0.7190 std=0.0110  (0s, 0.0/120 min, 2054232 trials/hr)
[6] mean=0.7398 std=0.0050  (0s, 0.0/120 min, 2230322 trials/hr)
[7] mean=0.7280 std=0.0077  (0s, 0.0/120 min, 2247051 trials/hr)
[8] mean=0.7375 std=0.0043  (0s, 0.0/120 min, 2379241 trials/hr)
[9] mean=0.7368 std=0.0055  (0s, 0.0/120 min, 2491062 trials/hr)
[10] mean=0.7321 std=0.0061  (0s, 0.0/120 min, 2545231 trials/hr)
[11] mean=0.7379 std=0.0086  (0s, 0.0/120 min, 2564867 trials/hr)
[12] mean=0.7344 std=0.0039  (0s, 0.0/120 min, 2639909 trials/hr)
[13] mean=0.7405 std=0.0045  (0s, 0.0/120 min, 2693401 trials/hr)
[14] mean=0.7355 std=0.0055  (0s, 0.0/120 min, 2749246 trials/hr)
[15] mean=0.7343 std=0.0095  (0s, 0.0/120 min, 2743651 trials/hr)
[16] mean=0.7374 st

C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

[124] mean=0.7411 std=0.0041  (130s, 2.2/120 min, 3429 trials/hr)


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

[125] mean=0.7381 std=0.0041  (44s, 2.9/120 min, 2581 trials/hr)


...

[141] mean=0.7328 std=0.0048  (84s, 42.9/120 min, 197 trials/hr)


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

[142] mean=0.7337 std=0.0067  (90s, 44.4/120 min, 192 trials/hr)


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

[143] mean=0.7382 std=0.0056  (278s, 49.1/120 min, 175 trials/hr)


### Commit your trials before going further

```bash
git add data/processed/tuning_trials/
git commit -m "feat: lightgbm_v2 stage-1 trials, learning_rate <your band>"
git push
```

Then wait until all three of us have pushed. Stage 2 centres on the *merged* winner
across all three bands, so running it early would centre it on your band's local best.

## 4. Merge stage 1 across the team

In [ ]:
stage1 = tuning.load_trials("lightgbm_v2_stage1")
print(f"{len(stage1)} stage-1 trials merged, from owners: {sorted(stage1['owner'].unique())}")
print(stage1.head(10)[["owner", "mean", "std"]].to_string())

# The historical record must be untouched: 05's search used the key `lightgbm_stage1`
# and had exactly 8 trials. If this trips, the new key collided with the old glob.
assert len(tuning.load_trials("lightgbm_stage1")) == 8, "05_tuning's trial history changed"

center = stage1.iloc[0]["params"]
print(f"\nMerged winner ({stage1.iloc[0]['mean']:.4f}), centering stage 2 on:")
print(json.dumps(center, indent=2))
print(f"\n05's tuned LightGBM for reference: 0.7443")

In [ ]:
# Which knobs actually mattered? Rank correlation between each knob and CV score across
# every merged trial. This is what tells the report whether widening num_leaves and
# min_child_samples was justified, rather than asserting it.
knobs = pd.DataFrame(list(stage1["params"])).assign(mean=stage1["mean"].values)
numeric = knobs.select_dtypes("number").drop(columns="mean")
influence = (numeric.corrwith(knobs["mean"], method="spearman")
             .sort_values(key=abs, ascending=False).rename("spearman_vs_cv_f1"))
print(influence.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
influence.plot.barh(ax=ax)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Spearman correlation with CV Macro F1")
ax.set_title(f"Which LightGBM knobs move the score ({len(stage1)} merged stage-1 trials)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / "lightgbm_v2_knob_influence.png", dpi=120)
plt.show()

## 5. Stage 2 - local random search, all knobs at once

05's stage 2 grid-searched two knobs and froze the other five, which cannot find
interactions - and interactions are exactly what a boosting model's knobs have
(`learning_rate` x `n_estimators`, `num_leaves` x `min_child_samples`).

This instead draws random points from a shrunk box (+/-30%) around the merged winner,
varying **every** knob simultaneously. The three of us draw from the same box with
different `random_state` values, which is an equal three-way split of the same space and
merges identically.

In [ ]:
SHRINK = 0.30


def around(value, lo, hi, integer=False, floor=0.0):
    '''+/-30% box around `value`, clipped to [lo, hi].

    `floor` is a minimum half-width. Without it a knob whose stage-1 winner landed
    at exactly 0 - entirely possible for reg_alpha / reg_lambda, whose range starts
    there - would get a zero-width box and could never move off 0 again.
    '''
    half = max(abs(value) * SHRINK, floor)
    a, b = max(value - half, lo), min(value + half, hi)
    if integer:
        a, b = int(round(a)), int(round(b))
        return randint(a, max(b, a + 1))  # randint's hi is exclusive
    return uniform(a, max(b - a, 1e-9))


refined_dist = {
    "learning_rate": around(center["learning_rate"], 0.005, 0.35, floor=0.002),
    "n_estimators": around(center["n_estimators"], 100, 1200, integer=True),
    "num_leaves": around(center["num_leaves"], 8, 600, integer=True),
    "min_child_samples": around(center["min_child_samples"], 1, 90, integer=True, floor=2),
    "min_child_weight": around(center["min_child_weight"], 1e-4, 20, floor=1e-3),
    "subsample": around(center["subsample"], 0.5, 1.0, floor=0.05),
    "colsample_bytree": around(center["colsample_bytree"], 0.1, 1.0, floor=0.05),
    "max_bin": [63, 127, 255],
    "reg_alpha": around(center["reg_alpha"], 0.0, 3.0, floor=0.25),
    "reg_lambda": around(center["reg_lambda"], 0.0, 3.0, floor=0.25),
}

refined_sampler = ParameterSampler(refined_dist, n_iter=200, random_state=STAGE2_SEED)

stage2_mine = tuning.run_search(
    "lightgbm_v2_stage2", OWNER, build_lgbm, refined_sampler,
    X_dev, y_dev, cv, budget_seconds=STAGE2_BUDGET_S,
)
print(f"\n{len(stage2_mine)} stage-2 trials this run")

### Commit again, then pull everyone's stage-2 trials

```bash
git add data/processed/tuning_trials/
git commit -m "feat: lightgbm_v2 stage-2 trials, seed <yours>"
git push
```

## 6. The winner - persist params and a dev-fit model

In [ ]:
stage2 = tuning.load_trials("lightgbm_v2_stage2")
all_trials = (pd.concat([stage1, stage2], ignore_index=True)
              .sort_values("mean", ascending=False).reset_index(drop=True))

best = all_trials.iloc[0]
best_params = best["params"]
print(f"Best of {len(all_trials)} trials "
      f"(stage 1: {len(stage1)}, stage 2: {len(stage2)}) across "
      f"{sorted(all_trials['owner'].unique())}")
print(f"CV Macro F1 {best['mean']:.4f} (std {best['std']:.4f})")
print(f"05's tuned LightGBM:  0.7443   ->  delta {best['mean'] - 0.7443:+.4f}")
print("\nWinning hyperparameters:")
print(json.dumps(best_params, indent=2))

with open(paths.DATA_PROCESSED / "best_lightgbm_v2_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

# Dev-only fit, matching the convention of the other 05x notebooks. The full refit on
# all 20,000 rows happens in 10_stacked_ensemble alongside the other members.
paths.MODELS.mkdir(parents=True, exist_ok=True)
dev_fit = build_lgbm(best_params).fit(X_dev, y_dev)
joblib.dump(dev_fit, paths.MODELS / "best_lightgbm_v2_tuned.pkl")

with open(paths.MODELS / "best_lightgbm_v2_tuned.json", "w") as f:
    json.dump({"model": "lightgbm_v2", "params": best_params,
               "cv_mean_f1": best["mean"], "cv_std_f1": best["std"],
               "n_trials": len(all_trials),
               "owners": sorted(all_trials["owner"].unique().tolist())}, f, indent=2)

print("\nSaved best_lightgbm_v2_params.json, best_lightgbm_v2_tuned.pkl (+ .json)")

In [ ]:
# Did the widened dimensions earn it? Where the winner sits inside each stage-1 range.
# A value still pinned at a boundary means round 5 should widen that knob further.
BOUNDS = {"num_leaves": (15, 400), "min_child_samples": (2, 60),
          "n_estimators": (150, 900), "colsample_bytree": (0.2, 1.0),
          "subsample": (0.6, 1.0), "min_child_weight": (1e-3, 10)}

rows = []
for k, (lo, hi) in BOUNDS.items():
    v = best_params[k]
    rows.append({"knob": k, "lo": lo, "value": v, "hi": hi,
                 "pct_of_range": round(100 * (v - lo) / (hi - lo), 1),
                 "at_boundary": (v - lo) / (hi - lo) < 0.05 or (v - lo) / (hi - lo) > 0.95})
print(pd.DataFrame(rows).to_string(index=False))
print("\nAny at_boundary=True is a knob whose optimum may lie outside this search box "
      "- the same signature that motivated this notebook.")

## Discussion / carry-forward -> `10_stacked_ensemble.ipynb`

_Fill in once all three searches have merged._

- Merged trial count, and each owner's contribution.
- Winning CV Macro F1 vs 05's 0.7443. Anything under ~+0.004 is inside the CV std and
  should be reported as "no measurable gain from re-searching", not as an improvement.
- Which knobs the section-4 influence plot says actually mattered, and whether the two
  widened dimensions (`num_leaves`, `min_child_samples`) moved off their old boundaries.
- Any knob still pinned at a boundary in the section-6 table.

**Carry forward:** `best_lightgbm_v2_params.json` is the `lightgbm_v2` member of the
round-4 ensemble. Cliffton owns its out-of-fold generation in
`10_stacked_ensemble.ipynb` section 3.